## Gold Layer — Business-Ready Aggregations

The **Gold Layer** contains pre-computed, analytics-ready tables derived from the Silver layer.
All tables are Delta-managed, append-free, and rebuilt on each refresh.

| Asset | Type | Purpose |
|---|---|---|
| `dev.gold.sales` | VIEW | Full Silver pass-through — single source of truth |
| `dev.gold.sales_by_outlet` | TABLE | Revenue KPIs per outlet |
| `dev.gold.sales_by_item_type` | TABLE | Category-level sales and pricing analysis |
| `dev.gold.top_items_by_outlet` | TABLE | Top-10 revenue items ranked per outlet |
| `dev.gold.outlet_performance_scorecard` | TABLE | Outlet tier benchmarking (type × size × location) |
| `dev.gold.mrp_sales_analysis` | TABLE | Cross-analysis of price tier, visibility and fat content vs sales |

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dev MANAGED LOCATION 'abfss://gold@devstgaccdeprj.dfs.core.windows.net/';
CREATE SCHEMA IF NOT EXISTS dev.gold;

-- Pass-through view: full Silver data exposed as Gold entry point
CREATE OR REPLACE VIEW dev.gold.sales
COMMENT 'Full cleansed and enriched Silver sales data — source for all Gold aggregations'
AS
SELECT * FROM dev.silver.sales;

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.sales_by_outlet
COMMENT 'Sales revenue KPIs aggregated per outlet'
AS
SELECT
    Outlet_Identifier,
    Outlet_Type,
    Outlet_Size,
    Outlet_Location_Type,
    Outlet_Establishment_Year,
    Outlet_Age_Years,
    COUNT(*)                                                            AS total_items_stocked,
    COUNT(DISTINCT Item_Identifier)                                     AS unique_items,
    ROUND(SUM(Item_Outlet_Sales),  2)                                   AS total_sales,
    ROUND(AVG(Item_Outlet_Sales),  2)                                   AS avg_sales_per_item,
    ROUND(MAX(Item_Outlet_Sales),  2)                                   AS max_item_sales,
    ROUND(MIN(Item_Outlet_Sales),  2)                                   AS min_item_sales,
    ROUND(STDDEV(Item_Outlet_Sales), 2)                                 AS stddev_sales,
    ROUND(AVG(Item_MRP),           2)                                   AS avg_mrp,
    current_timestamp()                                                 AS _gold_timestamp
FROM dev.silver.sales
GROUP BY
    Outlet_Identifier, Outlet_Type, Outlet_Size,
    Outlet_Location_Type, Outlet_Establishment_Year, Outlet_Age_Years
ORDER BY total_sales DESC

In [0]:
%sql
SELECT * FROM dev.gold.sales_by_outlet

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.sales_by_item_type
COMMENT 'Sales performance aggregated by item category, fat content and price tier'
AS
SELECT
    Item_Type,
    Item_Fat_Content,
    MRP_Band,
    COUNT(*)                                        AS total_records,
    COUNT(DISTINCT Item_Identifier)                 AS unique_items,
    ROUND(SUM(Item_Outlet_Sales),  2)               AS total_sales,
    ROUND(AVG(Item_Outlet_Sales),  2)               AS avg_sales,
    ROUND(PERCENTILE(Item_Outlet_Sales, 0.5), 2)    AS median_sales,
    ROUND(AVG(Item_MRP),           2)               AS avg_mrp,
    ROUND(AVG(Item_Weight),        2)               AS avg_weight,
    ROUND(AVG(Item_Visibility),    4)               AS avg_visibility,
    current_timestamp()                             AS _gold_timestamp
FROM dev.silver.sales
GROUP BY Item_Type, Item_Fat_Content, MRP_Band
ORDER BY total_sales DESC

In [0]:
%sql
SELECT * FROM dev.gold.sales_by_item_type

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.top_items_by_outlet
COMMENT 'Top 10 revenue-generating items per outlet, ranked by item sales'
AS
WITH ranked AS (
    SELECT
        Outlet_Identifier,
        Outlet_Type,
        Outlet_Size,
        Outlet_Location_Type,
        Item_Identifier,
        Item_Type,
        Item_Fat_Content,
        ROUND(Item_MRP, 4)          AS Item_MRP,
        MRP_Band,
        ROUND(Item_Outlet_Sales, 4) AS Item_Outlet_Sales,
        Sales_Band,
        RANK() OVER (
            PARTITION BY Outlet_Identifier
            ORDER BY Item_Outlet_Sales DESC
        )                           AS sales_rank
    FROM dev.silver.sales
)
SELECT * FROM ranked
WHERE  sales_rank <= 10
ORDER BY Outlet_Identifier, sales_rank

In [0]:
%sql
SELECT * FROM dev.gold.top_items_by_outlet ORDER BY Outlet_Identifier, sales_rank

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.outlet_performance_scorecard
COMMENT 'Revenue and efficiency benchmarking grouped by outlet type, size and location tier'
AS
SELECT
    Outlet_Type,
    Outlet_Size,
    Outlet_Location_Type,
    COUNT(DISTINCT Outlet_Identifier)                                          AS outlet_count,
    COUNT(*)                                                                   AS total_items,
    ROUND(SUM(Item_Outlet_Sales), 2)                                           AS total_sales,
    ROUND(AVG(Item_Outlet_Sales), 2)                                           AS avg_sales_per_item,
    ROUND(SUM(Item_Outlet_Sales) / COUNT(DISTINCT Outlet_Identifier), 2)       AS avg_revenue_per_outlet,
    ROUND(AVG(Outlet_Age_Years), 1)                                            AS avg_outlet_age_years,
    current_timestamp()                                                        AS _gold_timestamp
FROM dev.silver.sales
GROUP BY Outlet_Type, Outlet_Size, Outlet_Location_Type
ORDER BY total_sales DESC

In [0]:
%sql
SELECT * FROM dev.gold.outlet_performance_scorecard

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.mrp_sales_analysis
COMMENT 'Cross-analysis of price tier, visibility band and fat content vs actual sales performance'
AS
SELECT
    MRP_Band,
    Sales_Band,
    Item_Visibility_Band,
    Item_Fat_Content,
    COUNT(*)                                                                   AS records,
    ROUND(AVG(Item_MRP), 2)                                                   AS avg_mrp,
    ROUND(AVG(Item_Outlet_Sales), 2)                                          AS avg_sales,
    ROUND(SUM(Item_Outlet_Sales), 2)                                          AS total_sales,
    ROUND(AVG(Item_Outlet_Sales) / NULLIF(AVG(Item_MRP), 0), 4)              AS avg_sales_to_mrp_ratio,
    current_timestamp()                                                       AS _gold_timestamp
FROM dev.silver.sales
GROUP BY MRP_Band, Sales_Band, Item_Visibility_Band, Item_Fat_Content
ORDER BY total_sales DESC

In [0]:
%sql
SELECT * FROM dev.gold.mrp_sales_analysis

In [0]:
assets = {
    "dev.gold.sales":                     "VIEW",
    "dev.gold.sales_by_outlet":           "TABLE",
    "dev.gold.sales_by_item_type":        "TABLE",
    "dev.gold.top_items_by_outlet":       "TABLE",
    "dev.gold.outlet_performance_scorecard": "TABLE",
    "dev.gold.mrp_sales_analysis":        "TABLE",
}

print("=== Gold Layer Catalog ===")
print(f"\n  {'Asset':<45} {'Type':<8} {'Rows':>8}   {'Cols':>5}")
print("  " + "-" * 72)
for name, kind in assets.items():
    df = spark.table(name)
    print(f"  {name:<45} {kind:<8} {df.count():>8,}   {len(df.columns):>5}")